# IVSurfaceExplorer: Math and Data Exploration

This notebook walks through the core logic of the project:
1. Black-Scholes pricing formula
2. Implied Volatility inversion
3. Surface interpolation
4. SVI parametric fitting

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime

# Add src to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.bs import bs_price, implied_vol
from src.data import fetch_amd_options
from src.surface import build_surface, deform_surface
from src.svi import fit_svi_surface, svi_w

## 1. Verify Black-Scholes
Let's test a simple ATM call: $S=100, K=100, T=1, r=0.05, \sigma=0.20$.

In [ ]:
S, K, T, r, sigma = 100, 100, 1, 0.05, 0.20
price = bs_price(S, K, T, r, sigma, 'C')
print(f"BS Price: {price:.4f}")

iv = implied_vol(price, S, K, T, r, 'C')
print(f"Inverted IV: {iv:.4f}")

## 2. Fetch Live Data
Pulling the latest snapshot for AMD.

In [ ]:
df = fetch_amd_options()
print(f"Fetched {len(df)} option rows.")
df.head()

## 3. Surface Fitting
Constructing the grid and interpolating.

In [ ]:
# Add IVs if not already present
df['our_iv'] = df.apply(lambda r: implied_vol(r.mid, r.spot, r.strike, r.time_to_expiry, 0.045, r.option_type), axis=1)
surface = build_surface(df)
print("Surface built successfully.")

## 4. SVI Fitting
Fitting the parametric model per maturity.

In [ ]:
svi_results = fit_svi_surface(df)
svi_results.head()